In [2]:
import json
import timeit
import torch

from pydantic import BaseModel, Field
from transformers import pipeline


class AnimalClassification(BaseModel):
    animal: str = Field(description="Animal name provided by the user")
    category: str = Field(description="Animal category selected from the fixed options")
    confidence: float = Field(description="Model's confidence score for the chosen category")


# Load once, outside the timed function — mirrors not re-downloading/reloading
# the model on every call, same spirit as `local_files_only=True` for MLX.
MODEL_NAME = "facebook/bart-large-mnli"
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device="mps",  # Apple Silicon GPU
)


def test():
    animal_name = "Crocodile"
    choices = ["Mammal", "Bird", "Reptile", "Amphibian", "Fish", "Other"]

    raw = classifier(
        animal_name,
        candidate_labels=choices,
        hypothesis_template="This animal, {}, belongs to the category of {{}}.".format(animal_name),
    )
    print(raw)

    result = AnimalClassification(
        animal=animal_name,
        category=raw["labels"][0],
        confidence=raw["scores"][0],
    )

    print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))


total_time = timeit.timeit(test, number=1)
print(f"Total time: {total_time:.2f} seconds")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'Crocodile', 'labels': ['Amphibian', 'Reptile', 'Mammal', 'Other', 'Fish', 'Bird'], 'scores': [0.4139096438884735, 0.22007861733436584, 0.18031762540340424, 0.12319891899824142, 0.05124926194548607, 0.011245881207287312]}
{
  "animal": "Crocodile",
  "category": "Amphibian",
  "confidence": 0.4139096438884735
}
Total time: 0.20 seconds
